# Choosing a hypothesis test
One dataset of 60 people (simulated, `data/people.csv`), and one test for each kind of question.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

people = pd.read_csv('data/people.csv')
people.head()

In [ ]:
# the two categorical columns
print(people['gender'].value_counts())
print(people['age_group'].value_counts())

## 1. One categorical column: one-sample proportion test
H0: the share of men is 0.5.

In [ ]:
n = len(people)
men = (people['gender'] == 'male').sum()
p_hat, pi0 = men / n, 0.5
# standard error under H0 uses the claimed pi0
se = np.sqrt(pi0 * (1 - pi0) / n)
z = (p_hat - pi0) / se
print(f'p_hat = {p_hat:.3f}, SE = {se:.4f}, z = {z:.2f}')
print('two-tailed p (z-test):', round(2 * stats.norm.sf(abs(z)), 3))
print('exact binomial test p:', round(stats.binomtest(men, n=n, p=pi0).pvalue, 3))

## 2. Two categorical columns: chi-square test of independence

In [ ]:
table = pd.crosstab(people['gender'], people['age_group'])
print(table)
result = stats.chi2_contingency(table)
print(f'chi2 = {result.statistic:.2f}, df = {result.dof}, p = {result.pvalue:.2f}')

## 3. One numerical column: one-sample t-test
H0: mean height is 1.55 m.

In [ ]:
h = people['height']
print(f'mean {h.mean():.4f}, s {h.std():.3f}')
print(stats.ttest_1samp(h, popmean=1.55))

## 4. Two numerical columns: correlation test

In [ ]:
def correlation_t(r, n):
    # t = r * sqrt(n - 2) / sqrt(1 - r^2), with n - 2 degrees of freedom
    t = r * np.sqrt(n - 2) / np.sqrt(1 - r ** 2)
    return t, 2 * stats.t.sf(abs(t), n - 2)

for r, n in [(0.30, 30), (0.30, 100)]:
    t, p = correlation_t(r, n)
    print(f'r = {r}, n = {n}: t = {t:.2f}, p = {p:.4f}')

In [ ]:
result = stats.pearsonr(people['height'], people['weight'])
print('r =', round(result.statistic, 3), ' p =', result.pvalue)
print('by hand:', correlation_t(result.statistic, len(people)))
adults = people[people['age_group'] == 'adult']
print('adults only: r =', round(stats.pearsonr(adults['height'], adults['weight']).statistic, 3))

## 5. Numerical + two groups: two-sample t-test (adult heights by gender)

In [ ]:
men_h = adults.loc[adults['gender'] == 'male', 'height']
women_h = adults.loc[adults['gender'] == 'female', 'height']
print(len(men_h), round(men_h.mean(), 3), len(women_h), round(women_h.mean(), 3))
print(stats.ttest_ind(men_h, women_h))

## 6. Numerical + three groups: one-way ANOVA (weight by age group)

In [ ]:
print(people.groupby('age_group')['weight'].mean().round(1))
groups = [g['weight'] for _, g in people.groupby('age_group')]
print(stats.f_oneway(*groups))